In [4]:
import re
from pathlib import Path

from pymongo import MongoClient


def load_env(path: str = ".env") -> dict[str, str]:
    env: dict[str, str] = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
    return env


def expand_vars(value: str, env: dict[str, str]) -> str:
    pattern = re.compile(r"\$\{([^}]+)\}")
    return pattern.sub(lambda m: env.get(m.group(1), m.group(0)), value)


env = load_env(".env")

# Usa directamente el string de Atlas desde .env
mongo_uri = env.get("MONGO_URI") or env.get("MONGODB_URI") or env.get("ATLAS_URI") or ""

if not mongo_uri:
    raise ValueError("No se encontro MONGO_URI/MONGODB_URI/ATLAS_URI en .env")

mongo_uri = expand_vars(mongo_uri, env)

client = MongoClient(mongo_uri, serverSelectionTimeoutMS=8000)
print(client.admin.command("ping"))
client.list_database_names()

{'ok': 1}


['sample_mflix', 'admin', 'local']

In [ ]:
db = client.nueva_bbdd_pymongo
users = db.users

In [7]:
client.list_database_names()


['sample_mflix', 'admin', 'local']

In [ ]:
users.insert_one({"nombre": "Juan", "edad": 25, "es_adulto": True})

InsertOneResult(ObjectId('6a262374de980d24e4c890a5'), acknowledged=True)

In [ ]:
list(users.find())  # all

[{'_id': ObjectId('6a262374de980d24e4c890a5'),
  'nombre': 'Juan',
  'edad': 25,
  'es_adulto': True}]

In [16]:
users.insert_many(
    [
        {"nombre": "Carlos", "edad": 12, "es_adulto": False},
        {"nombre": "Valentina", "edad": 18, "es_adulto": True},
    ]
)


InsertManyResult([ObjectId('6a2623f1de980d24e4c890a8'), ObjectId('6a2623f1de980d24e4c890a9')], acknowledged=True)

In [ ]:
users_results = users.find({}, {"_id": 0})
for user in users_results:
    print(user)


{'nombre': 'Juan', 'edad': 25, 'es_adulto': True}
{'nombre': 'Carlos', 'edad': 12, 'es_adulto': False}
{'nombre': 'Valentina', 'edad': 18, 'es_adulto': True}
{'nombre': 'Carlos', 'edad': 12, 'es_adulto': False}
{'nombre': 'Valentina', 'edad': 18, 'es_adulto': True}


In [25]:
users_results = users.find(
    {"$or": [{"nombre": "Juan"}, {"nombre": "Carlos"}]}, {"_id": 0}
)
for user in users_results:
    print(user)


{'nombre': 'Juan', 'edad': 25, 'es_adulto': True}
{'nombre': 'Carlos', 'edad': 12, 'es_adulto': False}
{'nombre': 'Carlos', 'edad': 12, 'es_adulto': False}


In [26]:
users.update_one({"nombre": "Juan"}, {"$set": {"edad": 30}})
users_results = users.find({"nombre": "Juan"}, {"_id": 0})
for user in users_results:
    print(user)

{'nombre': 'Juan', 'edad': 30, 'es_adulto': True}


In [30]:
db.list_collection_names()

['users']

In [31]:
# actualizando varios documentos
users.update_many({"es_adulto": True}, {"$set": {"edad": 30}})
users_results = users.find({"es_adulto": True}, {"_id": 0})
for user in users_results:
    print(user)

{'nombre': 'Juan', 'edad': 30, 'es_adulto': True}
{'nombre': 'Valentina', 'edad': 30, 'es_adulto': True}
{'nombre': 'Valentina', 'edad': 30, 'es_adulto': True}


In [ ]:
# borrando un documento (el primero que coincida)
users.delete_one({"nombre": "Juan"})

# borrando varios documentos (todos los que coincidan)
users.delete_many({})
users.insert_many(
    [
        {"nombre": "Juan", "edad": 25, "es_adulto": True},
        {"nombre": "Carlos", "edad": 12, "es_adulto": False},
        {"nombre": "Valentina", "edad": 18, "es_adulto": True},
    ]
)
# contando documentos
total_count = users.count_documents({})
print(total_count)

3


In [ ]:
# contar documentos que cumplan cierta condicion
count_no_adulto = users.count_documents({"es_adulto": False})
print(count_no_adulto)

1


In [47]:
import json
import random

# Generar 100 documentos para la colección "usuarios"
usuarios = []
nombres = [
    "Ana",
    "Luis",
    "Juan",
    "María",
    "Pedro",
    "Carmen",
    "Fernando",
    "Isabel",
    "Miguel",
    "Teresa",
]
apellidos = [
    "García",
    "Rodríguez",
    "Pérez",
    "Fernández",
    "López",
    "González",
    "Martínez",
    "Torres",
    "Sánchez",
    "Ramírez",
]
nombres_productos = [
    "Camiseta",
    "Pantalón",
    "Zapatos",
    "Reloj",
    "Sombrero",
    "Cinturón",
    "Gafas",
    "Bolso",
    "Zapatillas",
    "Corbata",
]
categorias = ["Ropa", "Accesorios", "Calzado", "Electrónica", "Joyas"]

for i in range(100):
    nombre_completo = random.choice(nombres) + " " + random.choice(apellidos)
    edad = random.randint(18, 65)
    email = nombre_completo.split()[0].lower() + str(i) + "@ejemplo.com"
    productos = random.sample(nombres_productos, random.randint(0, 8))
    usuarios.append(
        {
            "id": i + 1,
            "nombre": nombre_completo,
            "edad": edad,
            "productos": productos,
            "email": email,
        }
    )


# Modificar el formato de usuarios.json para que sea compatible con mongoimport
usuarios_str = "\n".join([json.dumps(usuario) for usuario in usuarios])

# Guardar el nuevo formato en un archivo
ruta_usuarios_modificado = "usuarios.json"
with open(ruta_usuarios_modificado, "w") as f:
    f.write(usuarios_str)

# Generar 100 documentos para la colección "productos"
productos = []

for i in range(100):
    nombre_producto = random.choice(nombres_productos)
    precio = round(random.uniform(10.0, 100.0), 2)
    categoria = random.choice(categorias)
    productos.append(
        {
            "id": i + 1,
            "nombre": nombre_producto,
            "precio": precio,
            "categoria": categoria,
        }
    )


# Modificar el formato de usuarios.json para que sea compatible con mongoimport
productos_str = "\n".join([json.dumps(producto) for producto in productos])

# Guardar el nuevo formato en un archivo
ruta_productos = "productos.json"
with open(ruta_productos, "w") as f:
    f.write(productos_str)


In [48]:
users.delete_many({})
users.insert_many(usuarios)
products = db.products
products.delete_many({})
products.insert_many(productos)
print("Cantidad de usuarios en la bd:", users.count_documents({}))
print("Cantidad de productos en la bd:", products.count_documents({}))

Cantidad de usuarios en la bd: 100
Cantidad de productos en la bd: 100


In [50]:
# 1Cantidad de personas que tengan como producto unas Zapatillas

ex1 = users.count_documents({"productos": "Zapatillas"})
print(ex1)

# 2Cantidad de personas entre 20 y 30 años

ex2 = users.count_documents({"edad": {"$gte": 20, "$lte": 30}})
print(ex2)

# 3Cantidad de personas que hayan comprado 3 productos
ex3 = users.count_documents({"productos": {"$size": 3}})
print(ex3)

# 4Cantidad de personas que no tengan productos
ex4 = users.count_documents({"productos": {"$size": 0}})
print(ex4)

37
24
11
12


In [51]:
list(users.find({"productos": {"$size": 0}}))


[{'_id': ObjectId('6a2628e0de980d24e4c8924c'),
  'id': 16,
  'nombre': 'María García',
  'edad': 31,
  'productos': [],
  'email': 'maría15@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c89253'),
  'id': 23,
  'nombre': 'Fernando González',
  'edad': 62,
  'productos': [],
  'email': 'fernando22@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c89268'),
  'id': 44,
  'nombre': 'Juan Torres',
  'edad': 29,
  'productos': [],
  'email': 'juan43@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c8926a'),
  'id': 46,
  'nombre': 'Ana García',
  'edad': 59,
  'productos': [],
  'email': 'ana45@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c8927c'),
  'id': 64,
  'nombre': 'Teresa Rodríguez',
  'edad': 43,
  'productos': [],
  'email': 'teresa63@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c8927d'),
  'id': 65,
  'nombre': 'Teresa Torres',
  'edad': 23,
  'productos': [],
  'email': 'teresa64@ejemplo.com'},
 {'_id': ObjectId('6a2628e0de980d24e4c89287'),
  'id': 75,
  

In [61]:
# las personas que tengan entre 15 a 25 años
# y no tengan Bolso, agregar el Bolso Gratis a la lista de productos.

users.update_many(
    {"edad": {"$gte": 15, "$lte": 25}, "productos": {"$ne": "Bolso"}},
    {"$push": {"productos": "Bolso Gratis"}},
)

# las personas que tengan 5 o mas productos colocar el campo
# de aplica_descuento como True

users.update_many(
    {"$expr": {"$gte": [{"$size": "$productos"}, 5]}},
    {"$set": {"aplica_descuento": True}},
)
list(users.find({"aplica_descuento": True}, {"_id": 0, "nombre": 1, "productos": 1}))

[{'nombre': 'Juan Sánchez',
  'productos': ['Gafas', 'Bolso', 'Zapatos', 'Camiseta', 'Cinturón']},
 {'nombre': 'Ana Torres',
  'productos': ['Pantalón', 'Cinturón', 'Corbata', 'Gafas', 'Zapatillas']},
 {'nombre': 'Fernando Torres',
  'productos': ['Camiseta',
   'Corbata',
   'Cinturón',
   'Zapatillas',
   'Reloj',
   'Zapatos',
   'Bolso Gratis',
   'Bolso Gratis',
   'Bolso Gratis',
   'Bolso Gratis',
   'Bolso Gratis']},
 {'nombre': 'Teresa Fernández',
  'productos': ['Bolso', 'Gafas', 'Reloj', 'Cinturón', 'Zapatillas']},
 {'nombre': 'Fernando Torres',
  'productos': ['Zapatillas', 'Reloj', 'Bolso', 'Sombrero', 'Zapatos']},
 {'nombre': 'Pedro Torres',
  'productos': ['Camiseta',
   'Bolso',
   'Sombrero',
   'Reloj',
   'Cinturón',
   'Zapatillas',
   'Pantalón',
   'Gafas']},
 {'nombre': 'Ana Ramírez',
  'productos': ['Camiseta',
   'Corbata',
   'Zapatillas',
   'Sombrero',
   'Gafas',
   'Cinturón',
   'Reloj']},
 {'nombre': 'María López',
  'productos': ['Pantalón',
   'Zapatos

In [66]:
# Version idempotente: evita duplicados de "Bolso Gratis" al re-ejecutar

# 1) Limpia duplicados existentes en el array productos
users.update_many(
    {},
    [{"$set": {"productos": {"$setUnion": ["$productos", []]}}}],
)

# 2) Agrega "Bolso Gratis" una sola vez por usuario
users.update_many(
    {"edad": {"$gte": 15, "$lte": 25}, "productos": {"$ne": "Bolso"}},
    {"$addToSet": {"productos": "Bolso Gratis"}},
)

# 3) Verificacion: nadie debe tener "Bolso Gratis" repetido
duplicados = list(
    users.find(
        {
            "$expr": {
                "$gt": [
                    {
                        "$size": {
                            "$filter": {
                                "input": "$productos",
                                "as": "p",
                                "cond": {"$eq": ["$$p", "Bolso Gratis"]},
                            }
                        }
                    },
                    1,
                ]
            }
        },
        {"_id": 0, "nombre": 1, "productos": 1},
    )
)
print("Usuarios con 'Bolso Gratis' duplicado:", len(duplicados))
duplicados[:5]

Usuarios con 'Bolso Gratis' duplicado: 0


[]

In [69]:
duplicados = list(
    users.aggregate(
        [
            # 1. Desarmamos el array de productos en filas individuales
            {"$unwind": "$productos"},
            # 2. Filtramos solo los que son "Bolso Gratis"
            {"$match": {"productos": "Bolso Gratis"}},
            # 3. Agrupamos por usuario y contamos cuántos bolsos gratis tiene cada uno
            {
                "$group": {
                    "_id": "$nombre",
                    "cantidad": {"$sum": 1},
                    "productos": {"$push": "$productos"},
                }
            },
            # 4. Nos quedamos solo con los que tengan más de 1
            {"$match": {"cantidad": {"$gt": 1}}},
        ]
    )
)
duplicados


[]